# Démo : un job MapReduce décortiqué

Les logs HTTP du serveur web du Kennedy Space Center (NASA), juillet 1995. Pour chaque code de réponse HTTP,
on compte les requêtes et les octets envoyés. Le même mapper et le même reducer, en Python, tournent d'abord
sur une machine, puis sur le cluster avec Hadoop Streaming.

**Critère de réussite** : chaque job du cluster donne exactement le résultat du même calcul en Python, sur une
seule machine.

Avant de commencer : le cluster tourne (`docker compose up -d` à la racine du module), et ce notebook est ouvert
depuis son dossier, avec le noyau `.venv` du module.

## Préparation

Une cellule qui commence par `%%client` s'exécute dans le conteneur client du cluster, où sont installés `hdfs`
et `mapred`. Le dossier de ce notebook y est monté sous `/cours/seance3/demo_job_decortique`.

In [1]:
import subprocess
from IPython.core.magic import register_cell_magic


@register_cell_magic
def client(ligne, cellule):
    """%%client : les commandes de la cellule s'exécutent dans le conteneur client du cluster."""
    commande = ["docker", "compose", "exec", "-T", "-w", "/cours/seance3/demo_job_decortique",
                "client", "bash", "-c", "set -e\n" + cellule]     # set -e : arrêt à la première erreur
    processus = subprocess.Popen(commande, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for sortie in processus.stdout:                                  # affichée au fil de l'eau
        print(sortie, end="")
    if processus.wait() != 0:
        raise RuntimeError("La commande a échoué : voir sa sortie ci-dessus")

Télécharger les logs de juillet 1995 (21 Mo compressés).

In [2]:
%%client
mkdir -p data
curl -sS https://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz | gunzip > data/jul95.log
ls -l data/jul95.log

-rw-r--r-- 1 spark spark 205242368 Nov 18  2026 data/jul95.log


Déposer le fichier dans HDFS, en blocs de 32 Mo : 7 blocs, donc 7 tâches map. Ses 10 premières lignes vont à
part : elles serviront à mesurer le coût fixe d'un job.

In [3]:
%%client
hdfs dfs -rm -r -f -skipTrash /demo3
hdfs dfs -mkdir -p /demo3/nasa /demo3/petit
hdfs dfs -D dfs.blocksize=32m -put data/jul95.log /demo3/nasa/
head -n 10 data/jul95.log | hdfs dfs -put - /demo3/petit/dix_lignes.log
hdfs fsck /demo3/nasa/jul95.log | grep -E "Total blocks|Average block replication"

Deleted /demo3


Connecting to namenode via http://master:9870/fsck?ugi=spark&path=%2Fdemo3%2Fnasa%2Fjul95.log


 Total blocks (validated):	7 (avg. block size 29320338 B)
 Average block replication:	3.0


## Le mapper et le reducer

`%%writefile` enregistre la cellule dans un fichier de ce dossier.

In [4]:
%%writefile mapper.py
import sys
champ = sys.argv[1]                        # "code" ou "hote"
IGNOREE = "reporter:counter:Demo,Lignes ignorees,1\n"
sys.stdin.reconfigure(encoding="latin-1")  # octets hors UTF-8
for ligne in sys.stdin:
    m = ligne.split()
    ok = len(m) >= 3 and m[-2].isdigit()
    if not ok or not (m[-1].isdigit() or m[-1] == "-"):
        sys.stderr.write(IGNOREE)          # compteur Hadoop
        continue
    octets = 0 if m[-1] == "-" else int(m[-1])
    cle = m[-2] if champ == "code" else m[0]
    print(f"{cle}\t1\t{octets}")

Overwriting mapper.py


In [5]:
%%writefile reducer.py
import sys
cle_courante, requetes, octets = None, 0, 0
for ligne in sys.stdin:
    cle, n, o = ligne.rstrip("\n").split("\t")
    if cle != cle_courante:              # changement de clé
        if cle_courante is not None:
            print(f"{cle_courante}\t{requetes}\t{octets}")
        cle_courante, requetes, octets = cle, 0, 0
    requetes += int(n)
    octets += int(o)
if cle_courante is not None:
    print(f"{cle_courante}\t{requetes}\t{octets}")

Overwriting reducer.py


## 1. Sur une machine

Les lignes 48 à 53 de juillet, celles de l'exercice, traversent le mapper, le tri, puis le reducer. Pas encore
de Hadoop : un pipe Unix. Chaque cellule ajoute une étape.

In [6]:
%%client
sed -n 48,53p data/jul95.log

205.189.154.54 - - [01/Jul/1995:00:01:06 -0400] "GET /cgi-bin/imagemap/countdown?99,176 HTTP/1.0" 302 110
205.189.154.54 - - [01/Jul/1995:00:01:08 -0400] "GET /shuttle/missions/sts-71/images/images.html HTTP/1.0" 200 7634
www-a1.proxy.aol.com - - [01/Jul/1995:00:01:09 -0400] "GET /shuttle/countdown/ HTTP/1.0" 200 3985
dd15-062.compuserve.com - - [01/Jul/1995:00:01:12 -0400] "GET /news/sci.space.shuttle/archive/sci-space-shuttle-22-apr-1995-40.txt HTTP/1.0" 404 -
205.212.115.106 - - [01/Jul/1995:00:01:13 -0400] "GET /shuttle/missions/sts-71/images/images.html HTTP/1.0" 200 7634
piweba3y.prodigy.com - - [01/Jul/1995:00:01:14 -0400] "GET /shuttle/technology/images/srb_mod_compare_3-small.gif HTTP/1.0" 200 55666


In [7]:
%%client
sed -n 48,53p data/jul95.log | python3 mapper.py code

302	1	110
200	1	7634
200	1	3985
404	1	0
200	1	7634
200	1	55666


In [8]:
%%client
sed -n 48,53p data/jul95.log | python3 mapper.py code | LC_ALL=C sort

200	1	3985
200	1	55666
200	1	7634
200	1	7634
302	1	110
404	1	0


In [9]:
%%client
sed -n 48,53p data/jul95.log | python3 mapper.py code | LC_ALL=C sort | python3 reducer.py

200	4	74919
302	1	110
404	1	0


## 2. Sur le cluster

Le même code, sur le mois entier, avec Hadoop Streaming et 3 reducers. La première ligne efface la sortie du job,
qui ne doit pas exister. `%%time` mesure la durée de la cellule, de bout en bout.

In [10]:
%%time
%%client
hdfs dfs -rm -r -f -skipTrash /demo3/sorties/code
mapred streaming \
  -D mapreduce.job.reduces=3 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py code" \
  -reducer "python3 reducer.py" \
  -input /demo3/nasa/jul95.log \
  -output /demo3/sorties/code

packageJobJar: [] [/opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.5.0.jar] /tmp/streamjob14000581785583569274.jar tmpDir=null
2026-11-18 20:47:26,611 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at master/172.24.0.2:8032
2026-11-18 20:47:26,687 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at master/172.24.0.2:8032


2026-11-18 20:47:26,806 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/spark/.staging/job_1795021959367_0037
2026-11-18 20:47:26,982 INFO mapred.FileInputFormat: Total input files to process : 1


2026-11-18 20:47:27,014 INFO mapreduce.JobSubmitter: number of splits:7
2026-11-18 20:47:27,094 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1795021959367_0037
2026-11-18 20:47:27,094 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-11-18 20:47:27,184 INFO conf.Configuration: resource-types.xml not found
2026-11-18 20:47:27,184 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-11-18 20:47:27,189 WARN mapred.YARNRunner: Configuration yarn.app.mapreduce.am.resource.memory-mb=512Mi is overriding the yarn.app.mapreduce.am.resource.mb=1536 configuration


2026-11-18 20:47:27,217 INFO impl.YarnClientImpl: Submitted application application_1795021959367_0037
2026-11-18 20:47:27,234 INFO mapreduce.Job: The url to track the job: http://master:8088/proxy/application_1795021959367_0037/
2026-11-18 20:47:27,234 INFO mapreduce.Job: Running job: job_1795021959367_0037


2026-11-18 20:47:31,308 INFO mapreduce.Job: Job job_1795021959367_0037 running in uber mode : false
2026-11-18 20:47:31,309 INFO mapreduce.Job:  map 0% reduce 0%


2026-11-18 20:47:37,424 INFO mapreduce.Job:  map 71% reduce 0%


2026-11-18 20:47:38,434 INFO mapreduce.Job:  map 100% reduce 0%


2026-11-18 20:47:41,454 INFO mapreduce.Job:  map 100% reduce 67%


2026-11-18 20:47:42,462 INFO mapreduce.Job:  map 100% reduce 100%
2026-11-18 20:47:42,471 INFO mapreduce.Job: Job job_1795021959367_0037 completed successfully
2026-11-18 20:47:42,523 INFO mapreduce.Job: Counters: 56
	File System Counters
		FILE: Number of bytes read=24254310
		FILE: Number of bytes written=51726359
		FILE: Number of read operations=0
		FILE: Number of large read operations=0
		FILE: Number of write operations=0
		HDFS: Number of bytes read=205267581
		HDFS: Number of bytes written=102
		HDFS: Number of read operations=36
		HDFS: Number of large read operations=0
		HDFS: Number of write operations=6
		HDFS: Number of bytes read erasure-coded=0
	Job Counters 
		Killed map tasks=1
		Launched map tasks=7
		Launched reduce tasks=3
		Data-local map tasks=7
		Total time spent by all maps in occupied slots (ms)=62632
		Total time spent by all reduces in occupied slots (ms)=14964
		Total time spent by all map tasks (ms)=31316
		Total time spent by all reduce tasks (ms)=7482
		

CPU times: user 13 ms, sys: 10.4 ms, total: 23.5 ms
Wall time: 18.2 s


Un fichier de sortie par reducer.

In [11]:
%%client
hdfs dfs -ls /demo3/sorties/code
for f in part-00000 part-00001 part-00002; do
  echo "--- $f"
  hdfs dfs -cat /demo3/sorties/code/$f
done

Found 4 items
-rw-r--r--   3 spark supergroup          0 2026-11-18 20:47 /demo3/sorties/code/_SUCCESS
-rw-r--r--   3 spark supergroup         63 2026-11-18 20:47 /demo3/sorties/code/part-00000
-rw-r--r--   3 spark supergroup          9 2026-11-18 20:47 /demo3/sorties/code/part-00001
-rw-r--r--   3 spark supergroup         30 2026-11-18 20:47 /demo3/sorties/code/part-00002


--- part-00000


200	1701534	38692291442
302	46573	3682049
404	10845	0
500	62	0


--- part-00001


501	14	0


--- part-00002


304	132627	0
400	5	0
403	54	0


## 3. Avec un combiner

Une ligne de plus : `-combiner`, avec le même reducer. Du journal, on ne garde que les compteurs utiles
(`grep`). Par code HTTP, puis par hôte, sans et avec combiner.

In [12]:
%%time
%%client
hdfs dfs -rm -r -f -skipTrash /demo3/sorties/code_combiner
mapred streaming \
  -D mapreduce.job.reduces=3 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py code" \
  -combiner "python3 reducer.py" \
  -reducer "python3 reducer.py" \
  -input /demo3/nasa/jul95.log \
  -output /demo3/sorties/code_combiner 2>&1 | grep -E "Combine|Reduce shuffle bytes|completed"

2026-11-18 20:48:05,518 INFO mapreduce.Job: Job job_1795021959367_0038 completed successfully
		Combine input records=1891714
		Combine output records=45
		Reduce shuffle bytes=778
CPU times: user 2.92 ms, sys: 8.45 ms, total: 11.4 ms
Wall time: 18.4 s


In [13]:
%%time
%%client
hdfs dfs -rm -r -f -skipTrash /demo3/sorties/hote
mapred streaming \
  -D mapreduce.job.reduces=3 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py hote" \
  -reducer "python3 reducer.py" \
  -input /demo3/nasa/jul95.log \
  -output /demo3/sorties/hote 2>&1 | grep -E "Reduce input groups|Reduce shuffle bytes|completed"

2026-11-18 20:48:27,007 INFO mapreduce.Job: Job job_1795021959367_0039 completed successfully
		Reduce input groups=81982
		Reduce shuffle bytes=53556194
CPU times: user 2.89 ms, sys: 8.07 ms, total: 11 ms
Wall time: 21.5 s


In [14]:
%%time
%%client
hdfs dfs -rm -r -f -skipTrash /demo3/sorties/hote_combiner
mapred streaming \
  -D mapreduce.job.reduces=3 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py hote" \
  -combiner "python3 reducer.py" \
  -reducer "python3 reducer.py" \
  -input /demo3/nasa/jul95.log \
  -output /demo3/sorties/hote_combiner 2>&1 | grep -E "Combine|Reduce shuffle bytes|completed"

2026-11-18 20:48:47,563 INFO mapreduce.Job: Job job_1795021959367_0040 completed successfully
		Combine input records=1891714
		Combine output records=111812
		Reduce shuffle bytes=3414865
CPU times: user 3.06 ms, sys: 8.68 ms, total: 11.7 ms
Wall time: 20.6 s


## 4. Une tâche échoue en plein job

Le job par code est relancé en arrière-plan. Dès qu'il tourne, `mapred job -fail-task` fait échouer la
première tentative de la tâche map 1. Ensuite, dans l'historique (http://localhost:19888) : ce job, Map tasks,
puis la tâche `m_000001`.

In [15]:
%%time
%%client
# 1. Le job démarre en arrière-plan (&), son journal va dans /tmp/panne.log
hdfs dfs -rm -r -f -skipTrash /demo3/sorties/code_panne
mapred streaming \
  -D mapreduce.job.reduces=3 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py code" \
  -reducer "python3 reducer.py" \
  -input /demo3/nasa/jul95.log \
  -output /demo3/sorties/code_panne > /tmp/panne.log 2>&1 &

# 2. Dès que le job tourne (« running in uber mode » dans son journal),
#    on fait échouer la première tentative (_0) de la tâche map 1 (m_000001)
until grep -q "uber mode" /tmp/panne.log; do sleep 1; done
JOB=$(grep -o -m 1 "job_[0-9]*_[0-9]*" /tmp/panne.log)
TENTATIVE=$(echo $JOB | sed "s/job/attempt/")_m_000001_0
mapred job -fail-task $TENTATIVE 2>/dev/null

# 3. On attend la fin du job, puis on lit son journal
wait
grep -E "Running job|completed successfully|Failed map tasks|Launched map tasks" /tmp/panne.log

Killed task attempt_1795021959367_0041_m_000001_0 by failing it


2026-11-18 20:48:50,528 INFO mapreduce.Job: Running job: job_1795021959367_0041
2026-11-18 20:49:08,816 INFO mapreduce.Job: Job job_1795021959367_0041 completed successfully
		Failed map tasks=1
		Launched map tasks=8
CPU times: user 3.84 ms, sys: 8.21 ms, total: 12 ms
Wall time: 21.3 s


## Le coût fixe d'un job

Le même job, sur les 10 premières lignes. Comparez sa durée à celle du job sur le mois entier.

In [16]:
%%time
%%client
hdfs dfs -rm -r -f -skipTrash /demo3/sorties/petit
mapred streaming \
  -D mapreduce.job.reduces=3 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py code" \
  -reducer "python3 reducer.py" \
  -input /demo3/petit/dix_lignes.log \
  -output /demo3/sorties/petit 2>&1 | grep -E "number of splits|Map input records|completed"

2026-11-18 20:49:12,256 INFO mapreduce.JobSubmitter: number of splits:2
2026-11-18 20:49:25,665 INFO mapreduce.Job: Job job_1795021959367_0042 completed successfully
		Map input records=10
CPU times: user 2.32 ms, sys: 7.64 ms, total: 9.96 ms
Wall time: 16.8 s


## Le critère de réussite

On rapatrie le résultat des cinq jobs sur la machine, puis on refait le calcul en Python, sur une seule
machine, avec les mêmes règles de lecture que le mapper.

In [17]:
%%client
for sortie in code code_combiner code_panne hote hote_combiner; do
  hdfs dfs -cat "/demo3/sorties/$sortie/part-*" > data/$sortie.txt
done
wc -l data/*.txt

      8 data/code_combiner.txt
      8 data/code_panne.txt
      8 data/code.txt
  81982 data/hote_combiner.txt
  81982 data/hote.txt
 163988 total


In [18]:
from collections import Counter

requetes = {"code": Counter(), "hote": Counter()}
octets = {"code": Counter(), "hote": Counter()}
for ligne in open("data/jul95.log", encoding="latin-1"):
    m = ligne.split()
    if len(m) >= 3 and m[-2].isdigit() and (m[-1].isdigit() or m[-1] == "-"):
        for champ, cle in (("code", m[-2]), ("hote", m[0])):
            requetes[champ][cle] += 1
            octets[champ][cle] += 0 if m[-1] == "-" else int(m[-1])

for sortie in ["code", "code_combiner", "code_panne", "hote", "hote_combiner"]:
    champ = sortie.split("_")[0]
    attendu = sorted(f"{cle}\t{n}\t{octets[champ][cle]}" for cle, n in requetes[champ].items())
    obtenu = sorted(open(f"data/{sortie}.txt").read().splitlines())
    print(f"{sortie} : {len(obtenu)} clés, identique au calcul Python : {obtenu == attendu}")
    assert obtenu == attendu

code : 8 clés, identique au calcul Python : True
code_combiner : 8 clés, identique au calcul Python : True
code_panne : 8 clés, identique au calcul Python : True
hote : 81982 clés, identique au calcul Python : True
hote_combiner : 81982 clés, identique au calcul Python : True
